Transformers Would not load for me if I didn't uninstall and reinstall

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Pip install transformers

In [ ]:
!pip install transformers

Loading the model directly using the HF api token.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

import os
from huggingface_hub import InferenceClient
os.environ["HF_TOKEN"] = ""
client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4.1 Sentiment State Extraction API key and Tokenizer

In [2]:
text_example = ["Apple stock rose after strong earnings","Guidance disappointed investors"]

results=[]
for text in text_example:
    r = client.text_classification(
        text,
        model="ProsusAI/finbert",
        top_k=3
    )
    results.append(r)

s_i_scores = []
HD_i_vector =[]

# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
for r in results:
 scores = {d["label"]: d["score"] for d in r}
 positive = scores["positive"]
 neutral = scores["neutral"]
 negative = scores["negative"]

 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)


print(HD_i_vector,"\n",s_i_scores)

[[0.9266610145568848, 0.04188281670212746, 0.031456105411052704], [0.06635522842407227, 0.06892205029726028, 0.864722728729248]] 
 [0.8952049091458321, -0.7983675003051758]


4.1 Using Pipeline

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)  #top_k=None give pos-nue-neg values


text_example = ["Apple stock rose after strong earnings","Guidance disappointed investors"]


results = pipe(text_example)


s_i_scores = []
HD_i_vector =[]


# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
#print(results)
for r in results:
 scores = {d["label"]: d["score"] for d in r}
 positive = scores["positive"]
 neutral = scores["neutral"]
 negative = scores["negative"]


 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)

print(HD_i_vector,"\n",s_i_scores)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[0.9266611337661743, 0.04188275337219238, 0.03145609796047211], [0.0663551315665245, 0.06892195343971252, 0.8647229075431824]] 
 [0.8952050358057022, -0.7983677759766579]


4.2

In [5]:
import numpy as np

s_off = []
current_sentiment = 0

for sentiment in s_i_scores:
  current_sentiment+=sentiment
  s_off.append(current_sentiment)

'''
s_off[t] is the cumulative sentiment magnitude at the t-th update
If s_i_scores = [.9,-.6,.4] s_off = [.9,.3,.7]
'''
print(s_off)

[0.8952050358057022, 0.09683725982904434]


4.3

In [6]:
'''Code from AI for esimating parameters based on live tweets
from tick.hawkes import HawkesExpKern

model = HawkesExpKern(decays=beta)
model.fit(tweet_timestamps)
This will estimate mu_soc and alpha'''

#Using arbitrary values for parameters
event_times = np.arange(len(s_i_scores))

mu_soc = .2
alpha = .8
beta = 1

lambda_soc= []

for t in event_times:
  excitation = 0

  #Integral
  for ti in event_times[:t]:
    excitation+= alpha*np.exp(-beta*(t-ti))

  intensity = mu_soc + excitation
  lambda_soc.append(intensity)
print(lambda_soc)


#Arbitarily defining Tau and Epsilon
tau = 5
epsilon = 10**-6

s_soc = []

for t in range(len(s_i_scores)):

  lower_bound = max(0,t-tau)

  window_scores = s_i_scores[lower_bound:t+1]

  num=sum(window_scores)
  den=len(window_scores)+epsilon

  s_soc.append(num/den)
print(s_soc)

[0.2, np.float64(0.4943035529371539)]
[0.8952041406015617, 0.04841860570521932]


5.1/5.2 (Completely Crutched AI)

In [11]:
import numpy as np

def compute_volatility(lambda_soc, s_soc, sigma_base=0.2, kappa=0.5):
    sigma_t = []

    for t in range(len(lambda_soc)):
        sigma = sigma_base + kappa * np.log(1 + lambda_soc[t]) * abs(s_soc[t])
        sigma_t.append(sigma)

    return sigma_t

#μ(t,Soff)
def compute_drift(s_off, mu_base=0.01, gamma=0.1):
    mu_t = []

    for t in range(len(s_off)):
        mu = mu_base + gamma * s_off[t]
        mu_t.append(mu)

    return mu_t

#dSt​=St​(μdt+σdW)
def simulate_price(S0, mu_t, sigma_t, dt=1, jump_prob=0.1, mu_J=0, sigma_J=0.05):
    S = [S0]

    for t in range(1, len(mu_t)):
        dW = np.random.normal(0, np.sqrt(dt))

        drift = mu_t[t] * dt
        diffusion = sigma_t[t] * dW

        # Jump component
        jump = 0
        if np.random.rand() < jump_prob:
            J = np.random.normal(mu_J, sigma_J)
            jump = np.exp(J) - 1

        S_new = S[-1] * (1 + drift + diffusion + jump)
        S.append(S_new)

    return S


sigma_t = compute_volatility(lambda_soc, s_soc)

# Step 2: Drift
mu_t = compute_drift(s_off)

# Step 3: Price simulation
S = simulate_price(S0=100, mu_t=mu_t, sigma_t=sigma_t)

S

[100, np.float64(134.79801148611023)]

6.1

In [12]:
gamma = 0.5  # memory decay parameter

V_soc = []

for t in range(len(s_soc)):
    velocity = 0

    for k in range(t):
        weight = np.exp(-gamma * (t - k))
        dS = s_soc[k+1] - s_soc[k]

        velocity += weight * dS

    V_soc.append(velocity)

print("V_soc:", V_soc)

V_soc: [0, np.float64(-0.5136013891157938)]


6.2

In [14]:
theta1 = 0.5
theta2 = 0.5

Z_short = []

for t in range(len(S)):

    if t == 0:
        Z_short.append(0)
        continue

    delta_P = S[t] - S[t-1]
    delta_sigma = sigma_t[t] - sigma_t[t-1]

    Z = V_soc[t] - theta1 * delta_P - theta2 * delta_sigma

    Z_short.append(Z)

print("Z_short:", Z_short)


#This is definition 6.1
Gamma_thresh = 0.5

signals = []

for z in Z_short:
    if z > Gamma_thresh:
        signals.append(1)   # short volatility signal
    else:
        signals.append(0)

print("Signals:", signals)

Z_short: [0, np.float64(-17.87666533631704)]
Signals: [0, 0]
